# 対訳コーパスを介して言い換えを集める（ピボット）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_lexical_semantics/paraphrase_pivot.ipynb)

NLP クックブック 19-05 ／ 語の意味と意味の変化

「同じ外国語に訳される表現どうしは、言い換えである可能性が高い」という考え方で、対訳コーパスから言い換えの候補を集めたい。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import math, re
from collections import Counter, defaultdict
from datasets import load_dataset
from fugashi import Tagger

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja", split="train")
tagger = Tagger()
pairs = []
for ex in ds.select(range(200000)):
    ja, en = ex["translation"]["ja"].strip(), ex["translation"]["en"].strip().lower()
    if 0 < len(ja) <= 40 and 0 < len(en.split()) <= 15 and re.search(r"[ぁ-んァ-ヶ一-龥]", ja):
        ja_words = [(w.feature.lemma or w.surface).split("-")[0] for w in tagger(ja) if w.feature.pos1 in ("名詞", "動詞", "形容詞", "形状詞", "副詞")]
        en_words = re.findall(r"[a-z']+", en)
        if ja_words and en_words:
            pairs.append((ja_words, en_words))
print(len(pairs), "文対", pairs[0])

175212 文対 (['ラウール', 'ラウール', 'ビンセント', 'ハンナ'], ['yeah', 'vincent', 'hanna'])


In [3]:
# IBM モデル 1 で単語の翻訳確率 t(英語 | 日本語) を推定する (EM アルゴリズム)
def ibm_model1(pairs, iterations=5):
    t = {}
    for it in range(iterations):
        default = 1.0 if it == 0 else 1e-6          # 最初はすべて同じ確率。2 回目以降、表から落とした組にはごく小さい確率を与える
        count, total = defaultdict(float), defaultdict(float)
        for src, tgt in pairs:
            for e in tgt:
                probs = [t.get((e, f), default) for f in src]
                z = sum(probs)                      # e がどの f から訳されたかの確率を、現在の t で按分する
                for f, p in zip(src, probs):
                    count[e, f] += p / z; total[f] += p / z
        t = {(e, f): c / total[f] for (e, f), c in count.items() if c / total[f] > 1e-4}      # 確率の小さい組は捨てて、表を小さく保つ
    return t

t_en_ja = ibm_model1(pairs)                                  # P(英語 | 日本語)
t_ja_en = ibm_model1([(en, ja) for ja, en in pairs])         # P(日本語 | 英語)

def translations(table, word, k=6):
    return sorted(((round(p, 2), a) for (a, b), p in table.items() if b == word), reverse=True)[:k]

for w in ["嬉しい", "素晴らしい", "始める", "病院", "多分"]:
    print(w, translations(t_en_ja, w))

嬉しい [(0.11, 'glad'), (0.09, 'you'), (0.09, 'to'), (0.09, "i'm"), (0.05, 'happy'), (0.04, 'nice')]
素晴らしい [(0.09, 'great'), (0.07, 'wonderful'), (0.06, 'a'), (0.04, 'is'), (0.04, 'excellent'), (0.03, 'was')]
始める [(0.1, 'to'), (0.1, 'start'), (0.08, 'started'), (0.05, 'you'), (0.05, 'we'), (0.04, 'the')]
病院 [(0.29, 'hospital'), (0.13, 'the'), (0.08, 'to'), (0.08, 'a'), (0.06, 'in'), (0.04, 'at')]
多分 [(0.3, 'maybe'), (0.09, 'i'), (0.06, 'probably'), (0.04, 'think'), (0.02, 'you'), (0.02, 'perhaps')]


In [4]:
# ピボット: P(言い換え | 元の語) = Σ_英語 P(言い換え | 英語) × P(英語 | 元の語)
by_ja, by_en = defaultdict(dict), defaultdict(dict)
for (e, f), p in t_en_ja.items():
    by_ja[f][e] = p
for (f, e), p in t_ja_en.items():
    by_en[e][f] = p
freq = Counter(w for ja, _ in pairs for w in ja)
# 機能語を経由すると、どんな語とでも結び付いてしまう。英語の機能語と、日本語の軽い動詞・形式名詞は使わない
EN_STOP = set("a an the to of in on at for with by from and or but is am are was were be been being i you he she it we they me him her us them my your his its our their this that these those what which who do does did not no so if then just very can will would should could have has had there here".split())
JA_STOP = {"為る", "居る", "有る", "無い", "事", "成る", "良い", "そう", "行く", "言う", "呉れる", "物", "此の", "其の", "何", "様", "所", "今", "遣る", "来る", "見る", "思う", "知る"}

def paraphrases(word, k=8, min_freq=20):
    score = Counter()
    for e, p1 in by_ja[word].items():
        if e in EN_STOP:
            continue
        for f, p2 in by_en[e].items():
            if f != word and f not in JA_STOP and freq[f] >= min_freq:
                score[f] += p1 * p2
    return [(w, round(s, 3)) for w, s in score.most_common(k)]

for w in ["嬉しい", "素晴らしい", "始める", "病院", "多分", "怖い", "話す", "綺麗", "仕事", "直ぐ"]:
    print(w, "→", paraphrases(w))

嬉しい → [('幸せ', 0.019), ('本当', 0.012), ('過ぎる', 0.01), ('喜ぶ', 0.009), ('会う', 0.007), ('素敵', 0.007), ('もう', 0.007), ('分かる', 0.006)]
素晴らしい → [('凄い', 0.033), ('素敵', 0.017), ('どう', 0.011), ('偉大', 0.011), ('最高', 0.01), ('本当', 0.007), ('迚も', 0.006), ('分かる', 0.006)]
始める → [('開始', 0.017), ('始まる', 0.015), ('先ず', 0.009), ('最初', 0.007), ('会う', 0.007), ('始まり', 0.006), ('どう', 0.006), ('時', 0.006)]
病院 → [('入院', 0.029), ('必要', 0.014), ('運ぶ', 0.004), ('医者', 0.004), ('ブロック', 0.003), ('医療', 0.003), ('数', 0.003), ('先', 0.003)]
多分 → [('知れる', 0.041), ('恐らく', 0.034), ('分かる', 0.013), ('考える', 0.009), ('若し', 0.005), ('急度', 0.004), ('もう', 0.003), ('どう', 0.003)]
怖い → [('恐れる', 0.041), ('残念', 0.01), ('心配', 0.01), ('怯える', 0.01), ('恐ろしい', 0.008), ('分かる', 0.006), ('恐怖', 0.005), ('少し', 0.005)]
話す → [('話', 0.049), ('教える', 0.012), ('伝える', 0.008), ('聞く', 0.007), ('つく', 0.007), ('欲しい', 0.007), ('分かる', 0.005), ('どう', 0.004)]
綺麗 → []
仕事 → [('働く', 0.028), ('旨い', 0.013), ('ビジネス', 0.007), ('どう', 0.004), ('動く', 0.003), ('良く', 0.00

In [5]:
# 誤りの原因を見る: どの英語を経由したか
def explain(word, para):
    return sorted(((round(p1 * by_en[e].get(para, 0), 4), e) for e, p1 in by_ja[word].items() if para in by_en[e]), reverse=True)[:4]

for word, para in [("嬉しい", "幸せ"), ("話す", "言う"), ("怖い", "恐れる")]:
    print(word, "→", para, "経由した英語:", explain(word, para))

嬉しい → 幸せ 経由した英語: [(0.0189, 'happy'), (0.0001, 'deserve'), (0.0, 'would'), (0.0, 'was')]
話す → 言う 経由した英語: [(0.0228, 'tell'), (0.0126, 'told'), (0.0048, 'say'), (0.0045, 'you')]
怖い → 恐れる 経由した英語: [(0.0243, 'afraid'), (0.0093, 'scared'), (0.0057, 'fear'), (0.0005, 'frightened')]


- 2 つの表現が同じ外国語の表現に訳されるなら、それらは言い換えの候補になる（バイリンガル・ピボッティング）。英語の PPDB など、大規模な言い換えデータベースがこの方法で作られた。
- ここでは単語アライメントの最も単純なモデル（IBM モデル 1）で単語の翻訳確率を求めた。句の言い換えを集めるなら、機械翻訳の章の Moses が作るフレーズテーブルを使う。
- 機能語（英語の to、the、日本語の「為る」「居る」）は、どの語の翻訳候補にも顔を出すので、そのまま計算すると言い換えの上位を占めてしまう。両側の機能語を除くのが必須。
- 多義語を経由すると誤る（英語の 1 語が複数の意味を持つと、無関係な語どうしが結び付く）。反義語や、上位語・下位語も混ざりやすい。経由した語を確認すると原因が分かる。
- 得られるのは「候補」であり、文脈によって言い換えられるかどうかは変わる。単語ベクトルの類似度や言語モデルのスコアと組み合わせて絞り込む。
- 言い換えは、テキスト平易化（難しい語をやさしい語に）、検索のクエリ拡張、データ拡張、評価指標（言い換えを正解と認める）に使われる。